In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("ARTICULOS_LANGSMITH")
os.environ["LANGCHAIN_PROJECT"] = "Autores de articulos"
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = os.getenv("GITHUB_TOKEN")

from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
import sqlite3
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

/home/santi/Documentos/LangGraph/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/home/santi/Documentos/LangGraph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### State

In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

### MCP Servers

In [3]:
system_env = dict(os.environ)

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/santi/Documentos/LangGraph/"],
            "transport": "stdio",
            "env": system_env
        },
    }
)

In [12]:
async def run_agent():
    
    async with AsyncSqliteSaver.from_conn_string("filesystem_agent_history.db") as sqlite_saver:
        # 2. ENTER THE CONTEXT (The specific server room)
        # Instead of 'async with client', we use 'client.session("server_name")'
        async with client.session("filesystem") as session:
            
            # 3. LOAD TOOLS from that active session
            tools = await load_mcp_tools(session)
            model = ChatGroq(model="llama-3.1-8b-instant", temperature=0).bind_tools(tools)      
            
            print(f"Active tools: {[t.name for t in tools]}")

            base_path = "/home/santi/Documentos/LangGraph/"
            
            async def call_model(state: State):
                sys_msg = (
                    "system", 
                    "You are a technical assistant specialized in filesystem navigation.\n"
                    f"Your goal is to query and report the contents of the directory '{base_path}'.\n\n"
                    "Execution Instructions:\n"
                    "1. Review the message history of the current conversation.\n"
                    "2. If there is no previous message with tool results, call the tool to list files using exactly the indicated path.\n"
                    "3. If the files have already been listed in the history, do not—under any circumstances—invoke the tool again. Proceed directly to write a natural language summary of the files found and end the conversation.\n\n"
                    "Ensure you output ONLY the function call or the final text, never both at once."
                )
                
                prompt = [sys_msg] + state["messages"]
                response = model.invoke(prompt)

                return {"messages": [response]}

            # Graph Construction 
            workflow = StateGraph(State)
            workflow.add_node("agent", call_model)
            workflow.add_node("tools", ToolNode(tools)) # MCP tools are executed here

            workflow.add_edge(START, "agent")
            
            # Conditional logic to use tools
            def should_continue(state: State):
                if state["messages"][-1].tool_calls:
                    return "tools"
                return END

            workflow.add_conditional_edges("agent", should_continue)
            workflow.add_edge("tools", "agent")

            config = {"configurable": {"thread_id": "2"}}
            app = workflow.compile(checkpointer=sqlite_saver)

            # Execution
            inputs = {"messages": [("user", f"List the files in the directory {base_path}")]}        # IMPORTANT: The invocation occurs WITHIN the 'async with'
            result = await app.ainvoke(inputs, config=config) # Asynchronous graph invocation
            
            # for msg in result["messages"]:
            #     msg.pretty_print()

            # Listar el historial de estados
            async def print_history():
                print(f"--- Historial del Thread: {config['configurable']['thread_id']} ---")
                async for state in app.aget_state_history(config):
                    print(f"\nID: {state.config['configurable']['checkpoint_id']}")
                    print(f"Próximo nodo: {state.next}")
                    print(f"Mensajes: {len(state.values.get('messages', []))}")
                    print(f"Último mensaje: {state.values.get('messages')[-1].content if state.values.get('messages') else 'N/A'}")
                    print("-" * 40)

            await print_history()

# Execution in Notebook
await run_agent()

Active tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']
--- Historial del Thread: 2 ---

ID: 1f149386-bfc4-6e43-8003-0faff34bdd81
Próximo nodo: ()
Mensajes: 4
Último mensaje: The directory '/home/santi/Documentos/LangGraph/' contains several files and subdirectories. The files present are '.env', '.gitignore', 'README_LangGraph.md', 'bots_report.txt', and 'test.txt'. The subdirectories are '.git', '.venv', 'First Proyects', 'MCP', and 'Multi-Services Router'.
----------------------------------------

ID: 1f149386-ba5d-67ab-8002-13c8257d8387
Próximo nodo: ('agent',)
Mensajes: 3
Último mensaje: [{'type': 'text', 'text': '[FILE] .env\n[DIR] .git\n[FILE] .gitignore\n[DIR] .venv\n[DIR] First Proyects\n[DIR] MCP\n[DIR] Multi-Services Router\n[FILE] README_LangGraph.md\n[FILE] b